# Explainability Evaluation - Sensitivity

## Testing the Attention based Explainability and SHAP on Sensitivity

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Connect and Access Git Repository

import os
import sys
import json
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"

token = getpass("GitHub token: ")

REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
Cloning into '/content/bachelor-thesis'...
remote: Enumerating objects: 587, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 587 (delta 59), reused 48 (delta 25), pack-reused 485 (from 2)
Receiving objects: 100% (587/587), 36.19 MiB | 27.09 MiB/s, done.
Resolving deltas: 100% (299/299), done.
/content/bachelor-thesis
Repo exists: True


In [4]:
!git pull

Already up to date.


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))


from utils.dataset import PTBXLDataset
from utils.seed import set_seed, RANDOM_SEED
from explainability.attention import AnoFormerExplainability, TranADExplainability
from models.anoformer import AnoFormerMainModel
from models.tranad import TranADMainModel
from explainability.evaluation import SensitivityEvaluation


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [4]:
# Paths to Directories
# Load processed data and pretrained models

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

MODEL_DIR = RESULTS_DIR / "models"
ANOFORMER_MODEL_PATH = MODEL_DIR / "anoformer" / "anoformer_40epochs_larger_capacity_2.pt"
TRANAD_MODEL_PATH = MODEL_DIR / "tranad" / "tranad_20epochs_larger_capacity.pt"

SENSITIVITY_DIR = RESULTS_DIR / "evaluation" / "sensitivity"
SENSITIVITY_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Load my preprocessed training and test datasets

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2198, 1000, 12) (2198,)


In [6]:
# select only normal ecgs from Training set
normal_train_mask = y_train == 0
X_train_normal = X_train[normal_train_mask]
y_train_normal = y_train[normal_train_mask]

# Datasets
train_dataset = PTBXLDataset(
    X_train_normal,
    y_train_normal,
    normalize=True
)
test_dataset = PTBXLDataset(
    X_test,
    y_test,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

In [7]:
# Load previosly chosen ECGs for Explaianbility and Evaluation

SharedSamples = np.load(
    RESULTS_DIR
    / "evaluation"
    / "shared_explanation_samples.npz"
)

NormalIndex = int(SharedSamples["normal_index"])
AnomalousIndex = int(SharedSamples["anomalous_index"])

NormalECG, NormalLabel = test_dataset[NormalIndex]
AnomalousECG, AnomalousLabel = test_dataset[AnomalousIndex]

NormalECG = NormalECG.unsqueeze(0)
AnomalousECG = AnomalousECG.unsqueeze(0)

print("Index of Normal ECG:", NormalIndex, "Label:", int(NormalLabel))
print("Index of Anomalous ECG:", AnomalousIndex, "Label:", int(AnomalousLabel))

print("Normal ECG shape:", NormalECG.shape)
print("Anomalous ECG shape:", AnomalousECG.shape)

Index of Normal ECG: 583 Label: 0
Index of Anomalous ECG: 495 Label: 1
Normal ECG shape: torch.Size([1, 1000, 12])
Anomalous ECG shape: torch.Size([1, 1000, 12])


In [8]:
# Create the Evaluator

SensitivityEvaluator = SensitivityEvaluation(
    top_fraction=0.10,
    epsilon=1e-8,
)

In [9]:
def make_single_ecg_loader(ecg):
    """
    Create a DataLoader containing exactly one ECG

    Expected ECG shape:
    - (1000, 12), or
    - (1, 1000, 12)
    """

    if isinstance(ecg, np.ndarray):
        ecg = torch.tensor(ecg, dtype=torch.float32)

    if ecg.ndim == 2:
        ecg = ecg.unsqueeze(0)

    dummy_label = torch.zeros(ecg.shape[0], dtype=torch.long)

    dataset = TensorDataset(ecg, dummy_label)

    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
    )

    return loader

# AnoFormer Evaluation

In [10]:
# Create AnoFormer Model

AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

# Load the pretrained model
AnoFormer.Load(str(ANOFORMER_MODEL_PATH))

# Create Explainer to extract Attention Values
AnoFormerExplainer = AnoFormerExplainability(AnoFormer)

# This Anomaly Threshold was calculated based on the evaluation set
AnoFormerChosenThreshold = 0.17710630595684052

Modell geladen von: /content/drive/MyDrive/ECG data/results/models/anoformer/anoformer_40epochs_larger_capacity_2.pt


In [11]:
def anoformer_score_function(ecg):
    """
    Return the AnoFormer anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = AnoFormer.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=AnoFormerChosenThreshold,
        NumMaskRuns=5,
        Seed=RANDOM_SEED,
    )

    return scores[0]

In [12]:
# Test the score function

normal_score = anoformer_score_function(NormalECG)
anomalous_score = anoformer_score_function(AnomalousECG)

print("Normal score:", normal_score)
print("Normal prediction:", int(normal_score >= AnoFormerChosenThreshold))

print("Anomalous score:", anomalous_score)
print("Anomalous prediction:", int(anomalous_score >= AnoFormerChosenThreshold))

Normal score: 0.13429444
Normal prediction: 0
Anomalous score: 0.37212962
Anomalous prediction: 1


### Attention Evaluation

In [14]:
# Extract Attention Values of Original ECGs

NormalAttentionImportance = AnoFormerExplainer.GetAttentionImportance(
    TimeSeriesWindow=NormalECG,
    BlockIndex=0,
    HeadIndex=None,
)

AnomalousAttentionImportance = AnoFormerExplainer.GetAttentionImportance(
    TimeSeriesWindow=AnomalousECG,
    BlockIndex=0,
    HeadIndex=None,
)

print("Normal attention shape:", NormalAttentionImportance.shape) # Must be 1000 because we use all values not intervals
print("Anomalous attention shape:", AnomalousAttentionImportance.shape)

Normal attention shape: (1000,)
Anomalous attention shape: (1000,)


In [15]:
# Evaluate the Attention Sensitivity

AnoFormerAttentionNormalSensitivity = SensitivityEvaluator.evaluate(
    ecg=NormalECG,
    temporal_importance=NormalAttentionImportance,
    score_function=anoformer_score_function,
)

AnoFormerAttentionAnomalousSensitivity = SensitivityEvaluator.evaluate(
    ecg=AnomalousECG,
    temporal_importance=AnomalousAttentionImportance,
    score_function=anoformer_score_function,
)

In [18]:
# Print the Results

AnoFormerAttentionSensitivityResults = pd.DataFrame([
    {
        "Model": "AnoFormer",
        "Explanation": "Attention",
        "Sample": "Normal",
        "Original Score": AnoFormerAttentionNormalSensitivity["original_score"],
        "Perturbed Score": AnoFormerAttentionNormalSensitivity["perturbed_score"],
        "Score Difference": AnoFormerAttentionNormalSensitivity["score_difference"],
        "Sensitivity Score": AnoFormerAttentionNormalSensitivity["sensitivity_score"],
    },
    {
        "Model": "AnoFormer",
        "Explanation": "Attention",
        "Sample": "Anomalous",
        "Original Score": AnoFormerAttentionAnomalousSensitivity["original_score"],
        "Perturbed Score": AnoFormerAttentionAnomalousSensitivity["perturbed_score"],
        "Score Difference": AnoFormerAttentionAnomalousSensitivity["score_difference"],
        "Sensitivity Score": AnoFormerAttentionAnomalousSensitivity["sensitivity_score"],
    },
])

AnoFormerAttentionSensitivityResults

,Model,Explanation,Sample,Original Score,Perturbed Score,Score Difference,Sensitivity Score
0,AnoFormer,Attention,Normal,0.134294,0.152278,0.017984,0.133915
1,AnoFormer,Attention,Anomalous,0.372130,0.298300,-0.073830,0.198398


In [19]:
# Save the Results

anoformer_attention_sensitivity_path = SENSITIVITY_DIR / "anoformer_attention_sensitivity_results.csv"

AnoFormerAttentionSensitivityResults.to_csv(
    anoformer_attention_sensitivity_path,
    index=False,
)

print("Saved to:", anoformer_attention_sensitivity_path)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/sensitivity/anoformer_attention_sensitivity_results.csv


## SHAP Evaluation

In [13]:
anoFormer_shap_values = np.load(
    RESULTS_DIR / "shap" / "anoformer" / "anoformer_shap_values.npy"
)

In [14]:
def get_temporal_shap_importance_from_values(shap_values):
    """
    Convert SHAP values of shape (num_samples, 1000, 12)
    into timestep-level temporal importance.

    Output shape:
    - (num_samples, 1000)
    """

    shap_values = np.asarray(shap_values)

    if shap_values.ndim != 3:
        raise ValueError(
            f"Expected SHAP values with shape (num_samples, 1000, 12), "
            f"but got {shap_values.shape}."
        )

    if shap_values.shape[1:] != (1000, 12):
        raise ValueError(
            f"Expected SHAP values with shape (num_samples, 1000, 12), "
            f"but got {shap_values.shape}."
        )

    temporal_importance = np.abs(shap_values).sum(axis=2)

    return temporal_importance

In [15]:
SHAPImportance = get_temporal_shap_importance_from_values(anoFormer_shap_values)

NormalSHAPImportance = SHAPImportance[0]
AnomalousSHAPImportance = SHAPImportance[1]

print("Normal SHAP importance:", NormalSHAPImportance.shape)
print("Anomalous SHAP importance:", AnomalousSHAPImportance.shape)

Normal SHAP importance: (1000,)
Anomalous SHAP importance: (1000,)


In [16]:
AnoFormerSHAPNormalSensitivity = SensitivityEvaluator.evaluate(
    ecg=NormalECG,
    temporal_importance=NormalSHAPImportance,
    score_function=anoformer_score_function,
)

AnoFormerSHAPAnomalousSensitivity = SensitivityEvaluator.evaluate(
    ecg=AnomalousECG,
    temporal_importance=AnomalousSHAPImportance,
    score_function=anoformer_score_function,
)

In [17]:
AnoFormerSHAPSensitivityResults = pd.DataFrame([
    {
        "Model": "AnoFormer",
        "Explanation": "SHAP",
        "Sample": "Normal",
        "Original Score": AnoFormerSHAPNormalSensitivity["original_score"],
        "Perturbed Score": AnoFormerSHAPNormalSensitivity["perturbed_score"],
        "Score Difference": AnoFormerSHAPNormalSensitivity["score_difference"],
        "Sensitivity Score": AnoFormerSHAPNormalSensitivity["sensitivity_score"],
    },
    {
        "Model": "AnoFormer",
        "Explanation": "SHAP",
        "Sample": "Anomalous",
        "Original Score": AnoFormerSHAPAnomalousSensitivity["original_score"],
        "Perturbed Score": AnoFormerSHAPAnomalousSensitivity["perturbed_score"],
        "Score Difference": AnoFormerSHAPAnomalousSensitivity["score_difference"],
        "Sensitivity Score": AnoFormerSHAPAnomalousSensitivity["sensitivity_score"],
    },
])

AnoFormerSHAPSensitivityResults

,Model,Explanation,Sample,Original Score,Perturbed Score,Score Difference,Sensitivity Score
0,AnoFormer,SHAP,Normal,0.134294,0.087532,-0.046762,0.348206
1,AnoFormer,SHAP,Anomalous,0.372130,0.303904,-0.068225,0.183337


In [18]:
# Save the Results

anoformer_shap_sensitivity_path = SENSITIVITY_DIR / "anoformer_shap_sensitivity_results.csv"

AnoFormerSHAPSensitivityResults.to_csv(
    anoformer_shap_sensitivity_path,
    index=False,
)

print("Saved to:", anoformer_shap_sensitivity_path)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/sensitivity/anoformer_shap_sensitivity_results.csv


# TranAD Evaluation

In [19]:
# Create TranAD Model

TranAD = TranADMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    ForwardDimension=128,
    DropoutRate=0.1,
    LearningRate=1e-4,
    FocusGrowthRate=1.05,
    InitialFocusScore=1.0,
    MaxFocusScore=5.0,
    AnomalyPercentile=95.0,
)

# Load the pretrained model
TranAD.Load(str(TRANAD_MODEL_PATH))

# Create Explainer to extract Attention Values
TranADExplainer = TranADExplainability(TranAD)

# This Anomaly Threshold was calculated based on the evaluation set
TranADChosenThreshold = 0.010166172869503498


TranADModel: 268,376 Parameter | d_model=64, H=4, Enc=2, Dec×2=2
Model loaded  /content/drive/MyDrive/ECG data/results/models/tranad/tranad_20epochs_larger_capacity.pt


In [20]:
def tranad_score_function(ecg):
    """
    Return the TranAD anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = TranAD.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=TranADChosenThreshold,
    )

    return scores[0]

## Attention Evaluation

In [14]:
# Extract Attention Values of Original ECGs

TranADNormalAttentionImportance = TranADExplainer.GetAttentionImportance(
    TimeSeriesWindow=NormalECG,
    LayerIndex=0,
    HeadIndex=None,
)

TranADAnomalousAttentionImportance = TranADExplainer.GetAttentionImportance(
    TimeSeriesWindow=AnomalousECG,
    LayerIndex=0,
    HeadIndex=None,
)

print("Normal attention shape:", TranADNormalAttentionImportance.shape)
print("Anomalous attention shape:", TranADAnomalousAttentionImportance.shape)

Normal attention shape: (1000,)
Anomalous attention shape: (1000,)


In [15]:
# Evaluate the Attention Sensitivity

TranADAttentionNormalSensitivity = SensitivityEvaluator.evaluate(
    ecg=NormalECG,
    temporal_importance=TranADNormalAttentionImportance,
    score_function=tranad_score_function,
)

TranADAttentionAnomalousSensitivity = SensitivityEvaluator.evaluate(
    ecg=AnomalousECG,
    temporal_importance=TranADAnomalousAttentionImportance,
    score_function=tranad_score_function,
)

In [16]:
# Print the Results

TranADAttentionSensitivityResults = pd.DataFrame([
    {
        "Model": "TranAD",
        "Explanation": "Attention",
        "Sample": "Normal",
        "Original Score": TranADAttentionNormalSensitivity["original_score"],
        "Perturbed Score": TranADAttentionNormalSensitivity["perturbed_score"],
        "Score Difference": TranADAttentionNormalSensitivity["score_difference"],
        "Sensitivity Score": TranADAttentionNormalSensitivity["sensitivity_score"],
    },
    {
        "Model": "TranAD",
        "Explanation": "Attention",
        "Sample": "Anomalous",
        "Original Score": TranADAttentionAnomalousSensitivity["original_score"],
        "Perturbed Score": TranADAttentionAnomalousSensitivity["perturbed_score"],
        "Score Difference": TranADAttentionAnomalousSensitivity["score_difference"],
        "Sensitivity Score": TranADAttentionAnomalousSensitivity["sensitivity_score"],
    },
])

TranADAttentionSensitivityResults

,Model,Explanation,Sample,Original Score,Perturbed Score,Score Difference,Sensitivity Score
0,TranAD,Attention,Normal,0.007527,0.005735,-0.001792,0.238071
1,TranAD,Attention,Anomalous,0.065494,0.022672,-0.042822,0.653826


In [17]:
# Save the Results

tranad_attention_sensitivity_path = SENSITIVITY_DIR / "tranad_attention_sensitivity_results.csv"

TranADAttentionSensitivityResults.to_csv(
    tranad_attention_sensitivity_path,
    index=False,
)

print("Saved to:", tranad_attention_sensitivity_path)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/sensitivity/tranad_attention_sensitivity_results.csv


## SHAP Evaluation

In [21]:
tranad_shap_values = np.load(
    RESULTS_DIR / "shap" / "tranad" / "tranad_shap_values.npy"
)

In [22]:
SHAPImportance = get_temporal_shap_importance_from_values(tranad_shap_values)

NormalSHAPImportance = SHAPImportance[0]
AnomalousSHAPImportance = SHAPImportance[1]

print("Normal SHAP importance:", NormalSHAPImportance.shape)
print("Anomalous SHAP importance:", AnomalousSHAPImportance.shape)

Normal SHAP importance: (1000,)
Anomalous SHAP importance: (1000,)


In [23]:
TranADSHAPNormalSensitivity = SensitivityEvaluator.evaluate(
    ecg=NormalECG,
    temporal_importance=NormalSHAPImportance,
    score_function=anoformer_score_function,
)

TranADSHAPAnomalousSensitivity = SensitivityEvaluator.evaluate(
    ecg=AnomalousECG,
    temporal_importance=AnomalousSHAPImportance,
    score_function=anoformer_score_function,
)

In [24]:
TranADSHAPSensitivityResults = pd.DataFrame([
    {
        "Model": "TranAD",
        "Explanation": "SHAP",
        "Sample": "Normal",
        "Original Score": TranADSHAPNormalSensitivity["original_score"],
        "Perturbed Score": TranADSHAPNormalSensitivity["perturbed_score"],
        "Score Difference": TranADSHAPNormalSensitivity["score_difference"],
        "Sensitivity Score": TranADSHAPNormalSensitivity["sensitivity_score"],
    },
    {
        "Model": "TranAD",
        "Explanation": "SHAP",
        "Sample": "Anomalous",
        "Original Score": TranADSHAPAnomalousSensitivity["original_score"],
        "Perturbed Score": TranADSHAPAnomalousSensitivity["perturbed_score"],
        "Score Difference": TranADSHAPAnomalousSensitivity["score_difference"],
        "Sensitivity Score": TranADSHAPAnomalousSensitivity["sensitivity_score"],
    },
])

TranADSHAPSensitivityResults

,Model,Explanation,Sample,Original Score,Perturbed Score,Score Difference,Sensitivity Score
0,TranAD,SHAP,Normal,0.134294,0.144446,0.010152,0.075593
1,TranAD,SHAP,Anomalous,0.372130,0.291015,-0.081115,0.217975


In [25]:
# Save the Results

tranad_shap_sensitivity_path = SENSITIVITY_DIR / "tranad_shap_sensitivity_results.csv"

TranADSHAPSensitivityResults.to_csv(
    tranad_shap_sensitivity_path,
    index=False,
)

print("Saved to:", tranad_shap_sensitivity_path)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/sensitivity/tranad_shap_sensitivity_results.csv
